# 짱구 / 흰둥이 / 철수 YOLO fine-tuning + 객체 추적

**순서대로 위에서부터 ▶ 버튼을 누르기만 하면 됩니다.**

시작 전에: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장

## 1. 라이브러리 설치 & GPU 확인

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
ultralytics.checks()
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (런타임 유형을 T4 GPU로 바꾸세요)')

## 2. 파일 업로드
버튼이 나오면 **Roboflow에서 받은 zip 파일**(예: `jjanggu.v1i.yolov8.zip`)과 **`test_video.mp4`** 두 개를 같이 선택해서 올리세요.

In [ ]:
from google.colab import files
import glob, os, shutil, zipfile, yaml
up = files.upload()

# Roboflow zip 압축 풀기
zips = [f for f in up if f.endswith('.zip')]
assert zips, 'zip 파일이 없어요. Roboflow에서 받은 zip을 같이 올려주세요.'
DATA = '/content/jjanggu_dataset'
shutil.rmtree(DATA, ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall(DATA)

# 업로드한 영상 이름을 test_video.mp4 로 통일
vids = [f for f in up if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))]
if vids and vids[0] != 'test_video.mp4':
    shutil.move(vids[0], 'test_video.mp4')

# data.yaml 경로를 Colab 경로로 고치기
cfg = yaml.safe_load(open(f'{DATA}/data.yaml'))
cfg['path'] = DATA
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images' if os.path.isdir(f'{DATA}/valid') else 'train/images'
if os.path.isdir(f'{DATA}/test'):
    cfg['test'] = 'test/images'
else:
    cfg.pop('test', None)
yaml.safe_dump(cfg, open(f'{DATA}/data.yaml', 'w'), allow_unicode=True)

for s in ['train', 'valid', 'test']:
    if os.path.isdir(f'{DATA}/{s}/images'):
        print(s, len(os.listdir(f'{DATA}/{s}/images')), '장')
print(open(f'{DATA}/data.yaml').read())

## 3. 학습 (fine-tuning)
COCO로 사전학습된 `yolov8n.pt`를 가져와서 3개 부류로 추가 학습합니다. T4 GPU 기준 10분 안팎 걸려요.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')          # 사전학습 모델 (전이학습 시작점)
results = model.train(
    data='/content/jjanggu_dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    patience=30,
    name='jjanggu',
    exist_ok=True,
)

## 4. 학습 결과 / 성능 확인 (보고서용 그림)

In [ ]:
from IPython.display import Image, display
import os, glob
run = '/content/runs/detect/jjanggu'
for pat in ['results.png', '*confusion_matrix.png', '*PR_curve.png', 'val_batch0_pred.jpg']:
    for f in sorted(glob.glob(f'{run}/{pat}'))[:1]:
        print('==', os.path.basename(f)); display(Image(filename=f, width=900))

In [ ]:
from ultralytics import YOLO
best = YOLO('/content/runs/detect/jjanggu/weights/best.pt')
m = best.val(data='/content/jjanggu_dataset/data.yaml')
print(f'mAP50     : {m.box.map50:.3f}')
print(f'mAP50-95  : {m.box.map:.3f}')
print(f'Precision : {m.box.mp:.3f}')
print(f'Recall    : {m.box.mr:.3f}')
for i, name in best.names.items():
    print(f'  {name:10s} AP50 = {m.box.ap50[i]:.3f}')

## 5. 객체 감지 + 추적 영상 만들기
- `model.track()`으로 객체마다 ID를 붙여 추적
- 각 ID의 중심점을 이어 **궤적(trajectory)** 을 그림
- 매 프레임 좌표를 **`trajectory.csv`** 로 따로 저장 (슬라이드 51쪽 요구사항)
- 전체 영상 중 감지된 객체가 가장 많은 **5초 구간**을 골라 `result_5s.mp4`로 저장

In [ ]:
import cv2, csv, glob, numpy as np
from collections import defaultdict
from ultralytics import YOLO

best = YOLO('/content/runs/detect/jjanggu/weights/best.pt')

# 테스트 영상 찾기 (content 폴더 안/밖 어디에 있어도 찾음)
found = glob.glob('/content/**/test_video.mp4', recursive=True) + glob.glob('/test_video.mp4')
assert found, 'test_video.mp4를 못 찾았어요. 왼쪽 📁 파일 목록에 영상을 올려주세요.'
VIDEO = found[0]
print('영상:', VIDEO)

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(3)), int(cap.get(4))
cap.release()

NAME_COLORS = {'jjanggu': (0, 0, 255), 'heendungi': (255, 0, 255), 'cheolsu': (255, 128, 0)}  # 짱구 빨강, 흰둥이 분홍, 철수 파랑
COLORS = {i: NAME_COLORS.get(n, (0, 255, 0)) for i, n in best.names.items()}
track_hist = defaultdict(list)
frames, det_counts, frame_classes, rows = [], [], [], []

for f_idx, r in enumerate(best.track(source=VIDEO, stream=True, persist=True,
                                     tracker='bytetrack.yaml', conf=0.35, verbose=False)):
    img = r.orig_img.copy()
    n, seen = 0, set()
    if r.boxes is not None and r.boxes.id is not None:
        for box, tid, cls, conf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                       r.boxes.cls.int().cpu().tolist(), r.boxes.conf.cpu().tolist()):
            x1, y1, x2, y2 = box.astype(int)
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
            track_hist[tid].append((cx, cy))
            track_hist[tid] = track_hist[tid][-60:]          # 최근 60프레임 궤적만 표시
            c = COLORS[cls]
            cv2.rectangle(img, (x1, y1), (x2, y2), c, 2)
            cv2.putText(img, f'{best.names[cls]} #{tid} {conf:.2f}', (x1, max(15, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 2)
            pts = np.array(track_hist[tid], np.int32).reshape(-1, 1, 2)
            cv2.polylines(img, [pts], False, c, 3)
            cv2.circle(img, (cx, cy), 4, c, -1)
            rows.append([f_idx, round(f_idx / fps, 3), tid, best.names[cls], round(conf, 3), cx, cy, x1, y1, x2, y2])
            n += 1
            seen.add(cls)
    frames.append(img)
    det_counts.append(n)
    frame_classes.append(seen)

# 궤적 좌표 CSV 저장
with open('/content/trajectory.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['frame', 'time_s', 'track_id', 'class', 'conf', 'cx', 'cy', 'x1', 'y1', 'x2', 'y2'])
    w.writerows(rows)
print('총 프레임', len(frames), '/ 궤적 좌표', len(rows), '개 저장')
for i, name in best.names.items():
    print(f'  {name:10s}: {sum(i in s for s in frame_classes)} 프레임에서 감지')

def save(path, fr):
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    for x in fr: out.write(x)
    out.release()

save('/content/result_full.mp4', frames)

# 5초 구간: 감지된 객체가 가장 많은 구간
L = min(int(round(fps * 5)), len(frames))
scores = np.convolve(det_counts, np.ones(L), 'valid')
s = int(np.argmax(scores))
save('/content/result_5s.mp4', frames[s:s + L])
names_in = [best.names[i] for i in sorted(set().union(*frame_classes[s:s + L]))]
print(f'5초 구간: {s / fps:.1f}s ~ {(s + L) / fps:.1f}s  /  나온 캐릭터: {names_in}')

### 결과 영상 미리보기

In [ ]:
!ffmpeg -y -loglevel error -i /content/result_5s.mp4 -vcodec libx264 -pix_fmt yuv420p /content/result_5s_h264.mp4
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('/content/result_5s_h264.mp4', 'rb').read()).decode()
HTML(f'<video width=640 controls autoplay loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 6. 결과 파일 다운로드
`best.pt`(학습 모델), 결과 영상, 궤적 CSV, 학습 그래프를 한 번에 받아요. 이걸 GitHub에 올리면 됩니다.

In [ ]:
!mkdir -p /content/submit
!cp /content/runs/detect/jjanggu/weights/best.pt /content/submit/
!cp /content/result_5s_h264.mp4 /content/submit/result_5s.mp4
!cp /content/trajectory.csv /content/submit/
!cp /content/runs/detect/jjanggu/*.png /content/runs/detect/jjanggu/results.csv /content/submit/
!cd /content && rm -f submit.zip && zip -q -r submit.zip submit
from google.colab import files
files.download('/content/submit.zip')